<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 PhysicsIQ Reproduction with Cosmos Framework

This notebook walks through reproducing the PhysicsIQ benchmark with Cosmos3-Super using the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

PhysicsIQ has two task formats:

- **Image-to-Video (I2V):** condition on a single *switch frame* (the moment immediately before the physics event) and generate the next 5 seconds.
- **Video-to-Video (V2V):** condition on a 3-second clip leading up to the switch frame and generate the next 5 seconds.

We walk through one I2V case and one V2V case end-to-end (`0001_perspective-left_trimmed-ball-and-block-fall`). The full 198-case run is one cell at the end.

Reference scores for Cosmos3-Super with this exact recipe:

| Task | Cosmos3-Super PhysicsIQ score |
| --- | ---: |
| I2V | 43.8 |
| V2V | 59.7 |

## 1. Prerequisites

- Linux machine with NVIDIA GPU access (default recipe uses 4 GPUs)
- Model access on Hugging Face. Either run `uvx hf@latest auth login` or set `HF_TOKEN` in the environment.
- `uv >= 0.11.3` installed (https://docs.astral.sh/uv/getting-started/installation/).
- `ffmpeg` and `gcloud` (or `gsutil`) on PATH. The PhysicsIQ dataset is downloaded with the official Google Cloud Storage script.
- Cache/output paths with enough disk space. Full 198-case runs produce ~16 GB of raw outputs per task at 720p, 24 fps.

> **Headless servers:** if you see `libxcb.so.1: cannot open shared object file` when importing the model, install the system graphics libraries:
> ```bash
> apt-get install -y libxcb1 libgl1 libglib2.0-0
> ```

## 2. Configure Paths and Environment

All paths default to sensible locations under this `cosmos` checkout. Override any of them by exporting before launching the notebook:

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train   # or cu128-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-physicsiq
export COSMOS3_NUM_GPUS=4
export HF_HOME=/path/to/large/huggingface/cache    # Cosmos3-Super weights are ~90 GB
export UV_CACHE_DIR=/path/to/large/uv/cache          # defaults to $COSMOS3_REPO/.uv_cache
# Both default to a .hf_cache / .uv_cache folder inside $COSMOS3_REPO if not set.
export CUDA_VISIBLE_DEVICES=0,1,2,3
export COSMOS3_PHYSICSIQ_ROOT=/path/to/physics-IQ-benchmark
export COSMOS3_PHYSICSIQ_OUTPUT_ROOT=/path/to/physicsiq/outputs
```

In [ ]:
from pathlib import Path
import os
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos-framework"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "git@github.com:NVIDIA/cosmos-framework.git")
COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = Path(os.environ.get("UV_PROJECT_ENVIRONMENT", COSMOS3_REPO / ".venv")).resolve()
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "4")
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3")

PHYSICSIQ_ROOT = Path(
    os.environ.get(
        "COSMOS3_PHYSICSIQ_ROOT",
        COSMOS_ROOT / "evaluation" / "cosmos3" / "generator" / "physics_iq" / "physics-IQ-benchmark",
    )
).resolve()
PHYSICSIQ_NOTEBOOK_ROOT = COSMOS_ROOT / "evaluation" / "cosmos3" / "generator" / "physics_iq"
PHYSICSIQ_ASSETS = PHYSICSIQ_NOTEBOOK_ROOT / "assets"
PHYSICSIQ_OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_PHYSICSIQ_OUTPUT_ROOT", PHYSICSIQ_NOTEBOOK_ROOT / "outputs")
).resolve()
PHYSICSIQ_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEMO_CASE_ID = os.environ.get("COSMOS3_PHYSICSIQ_DEMO_CASE", "0001_perspective-left_trimmed-ball-and-block-fall")
CHECKPOINT = os.environ.get("COSMOS3_PHYSICSIQ_CHECKPOINT", "Cosmos3-Super")

MASTER_ADDR = os.environ.get("COSMOS3_MASTER_ADDR", "127.0.0.1")
MASTER_PORT_I2V = os.environ.get("COSMOS3_I2V_MASTER_PORT", free_local_port())
MASTER_PORT_V2V = os.environ.get("COSMOS3_V2V_MASTER_PORT", free_local_port())

for key, value in [
    ("COSMOS_ROOT", COSMOS_ROOT),
    ("COSMOS3_REPO", COSMOS3_REPO),
    ("COSMOS3_UV_ENV", COSMOS3_UV_ENV),
    ("COSMOS3_NUM_GPUS", COSMOS3_NUM_GPUS),
    ("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES),
    ("PHYSICSIQ_ROOT", PHYSICSIQ_ROOT),
    ("PHYSICSIQ_OUTPUT_ROOT", PHYSICSIQ_OUTPUT_ROOT),
    ("PHYSICSIQ_ASSETS", PHYSICSIQ_ASSETS),
    ("DEMO_CASE_ID", DEMO_CASE_ID),
    ("CHECKPOINT", CHECKPOINT),
]:
    print(f"{key}={value}")

# Export for the %%bash cells below.
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"] = str(COSMOS3_UV_ENV)
os.environ["UV_PROJECT_ENVIRONMENT"] = str(COSMOS3_UV_ENV)
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["PHYSICSIQ_ROOT"] = str(PHYSICSIQ_ROOT)
os.environ["PHYSICSIQ_OUTPUT_ROOT"] = str(PHYSICSIQ_OUTPUT_ROOT)
os.environ["PHYSICSIQ_ASSETS"] = str(PHYSICSIQ_ASSETS)
os.environ["DEMO_CASE_ID"] = DEMO_CASE_ID
os.environ["CHECKPOINT"] = CHECKPOINT
os.environ["COSMOS3_MASTER_ADDR"] = MASTER_ADDR
os.environ["COSMOS3_I2V_MASTER_PORT"] = MASTER_PORT_I2V
os.environ["COSMOS3_V2V_MASTER_PORT"] = MASTER_PORT_V2V

# Cache dirs — point these to a filesystem with plenty of space (e.g. lustre), not home.
# Cosmos3-Super downloads ~90 GB of weights into HF_HOME on first run.
HF_HOME = Path(os.environ.get("HF_HOME", COSMOS3_REPO / ".hf_cache")).resolve()
UV_CACHE_DIR = Path(os.environ.get("UV_CACHE_DIR", COSMOS3_REPO / ".uv_cache")).resolve()
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["UV_CACHE_DIR"] = str(UV_CACHE_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print(f"HF_HOME={HF_HOME}")
print(f"UV_CACHE_DIR={UV_CACHE_DIR}")

## 3. Clone or Reuse Cosmos Framework

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
git status --short --branch
git remote -v

## 4. Install Native PyTorch Dependencies

Installs framework dependencies with the requested CUDA group (default `cu130-train`).

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export GIT_LFS_SKIP_SMUDGE=1
cd "$COSMOS3_REPO"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
echo "Using UV_PROJECT_ENVIRONMENT=$UV_PROJECT_ENVIRONMENT"
uv sync --all-extras --group="$COSMOS3_UV_GROUP"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
  exit 1
fi

## 5. Verify GPU and Python Environment

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install Native PyTorch Dependencies cell first."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY

## 6. Download the PhysicsIQ Benchmark

PhysicsIQ is hosted by Google DeepMind at the public GCS bucket `gs://physics-iq-benchmark`. We use their official download helper, which fetches the 24 FPS split-videos, switch-frame JPGs, masks, descriptions, and scoring code.

Layout (under `$PHYSICSIQ_ROOT`):

```
physics-IQ-benchmark/
├── code/                                # official scorer (run_physics_iq.py, ...)
├── descriptions/                        # per-case captions
├── split-videos/
│   ├── conditioning/24FPS/              # 72-frame, 3-second conditioning mp4s (V2V inputs)
│   └── testing/24FPS/                   # 120-frame ground-truth mp4s (scoring targets)
├── switch-frames/                       # single JPG per case (I2V inputs)
└── video-masks/                         # binary motion masks for scoring
```

In [ ]:
%%bash
set -euo pipefail

if [ -d "$PHYSICSIQ_ROOT/split-videos/conditioning/24FPS" ] && [ -d "$PHYSICSIQ_ROOT/switch-frames" ]; then
  echo "PhysicsIQ benchmark already present at $PHYSICSIQ_ROOT"
  ls "$PHYSICSIQ_ROOT"
  exit 0
fi

if ! command -v gcloud >/dev/null 2>&1 && ! command -v gsutil >/dev/null 2>&1; then
  echo "Install gcloud SDK first: https://docs.cloud.google.com/sdk/docs/install-sdk" >&2
  exit 1
fi

PHYSICSIQ_REPO="$(dirname "$PHYSICSIQ_ROOT")/physics-IQ-benchmark-repo"
if [ ! -d "$PHYSICSIQ_REPO" ]; then
  git clone https://github.com/google-deepmind/physics-IQ-benchmark.git "$PHYSICSIQ_REPO"
fi

mkdir -p "$PHYSICSIQ_ROOT"
cd "$PHYSICSIQ_REPO"
# The official helper downloads from gs://physics-iq-benchmark into ./physics-IQ-benchmark/.
python3 ./code/download_physics_iq_data.py --output_dir "$PHYSICSIQ_ROOT" --fps 24 || \
  python3 ./code/download_physics_iq_data.py --fps 24

# If the helper wrote to ./physics-IQ-benchmark inside the repo, move it.
if [ -d "$PHYSICSIQ_REPO/physics-IQ-benchmark" ] && [ ! -e "$PHYSICSIQ_ROOT/split-videos" ]; then
  rsync -a "$PHYSICSIQ_REPO/physics-IQ-benchmark/" "$PHYSICSIQ_ROOT/"
fi

echo "--- contents of $PHYSICSIQ_ROOT ---"
ls "$PHYSICSIQ_ROOT"

## 7. Preview the Demo Case

We use `0001_perspective-left_trimmed-ball-and-block-fall` end-to-end: a ball and block held by overhead grippers are released and fall onto two pillows. The two inputs are the **switch frame** (instant of release) and the **3-second conditioning clip** that leads up to that instant, used for I2V and V2V respectively. 

In [ ]:
from IPython.display import Image, Video, display

switch_frame = (
    PHYSICSIQ_ROOT / "switch-frames" / f"{DEMO_CASE_ID.split('_', 1)[0]}_switch-frames_anyFPS_{DEMO_CASE_ID.split('_', 1)[1]}.jpg"
)
conditioning_72f = (
    PHYSICSIQ_ROOT / "split-videos" / "conditioning" / "24FPS"
    / f"{DEMO_CASE_ID.split('_', 1)[0]}_conditioning-videos_24FPS_{DEMO_CASE_ID.split('_', 1)[1]}_take-1_{DEMO_CASE_ID.split('_', 2)[-1]}.mp4"
)
# Some scenarios have alternate filename layouts; fall back to a glob if the exact name does not exist.
if not switch_frame.exists():
    candidates = sorted((PHYSICSIQ_ROOT / "switch-frames").glob(f"{DEMO_CASE_ID.split('_', 1)[0]}_*.jpg"))
    if candidates:
        switch_frame = candidates[0]
if not conditioning_72f.exists():
    candidates = sorted((PHYSICSIQ_ROOT / "split-videos" / "conditioning" / "24FPS").glob(
        f"{DEMO_CASE_ID.split('_', 1)[0]}_*.mp4"
    ))
    if candidates:
        conditioning_72f = candidates[0]

print("switch frame:", switch_frame)
print("conditioning video (72 frames @ 24 fps):", conditioning_72f)

display(Image(filename=str(switch_frame), width=480))
display(Video(filename=str(conditioning_72f), embed=True, width=480))

## 8. Helper Functions

Helpers used by both the I2V and V2V sections below:

- `case_paths(case_id)` — locate the switch frame JPG and 72-frame conditioning MP4 for a case.
- `build_i2v_input_jsonl(...)` — assemble a one-line JSONL row pointing at the switch frame.
- `build_v2v_input_jsonl(...)` — assemble a one-line JSONL row pointing at the 73-frame conditioning MP4.
- `prepare_v2v_conditioning(...)` — clone-pad the official 72-frame conditioning to 73 frames (the format the V2V model expects for Physics-IQ V2V).
- `stage_i2v_last120(...)` — drop the first frame of a 121-frame I2V output, keep the last 120.
- `stage_v2v_omit73(...)` — drop the 73-frame conditioning prefix of a 193-frame V2V output, keep the next 120.

All staging produces a 120-frame mp4 at 24 fps — the exact format the official PhysicsIQ scorer expects.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

I2V_PROMPTS_FILE = PHYSICSIQ_ASSETS / "i2v_prompts.json"
V2V_PROMPTS_FILE = PHYSICSIQ_ASSETS / "v2v_prompts.json"

I2V_PROMPTS = {row["name"]: row for row in json.loads(I2V_PROMPTS_FILE.read_text())}
V2V_PROMPTS = {row["name"]: row for row in json.loads(V2V_PROMPTS_FILE.read_text())}
assert len(I2V_PROMPTS) == 198 and len(V2V_PROMPTS) == 198

FFMPEG = shutil.which("ffmpeg")
assert FFMPEG, "ffmpeg is required on PATH"


def _ffmpeg(args: list[str]) -> None:
    subprocess.run([FFMPEG, "-y", "-loglevel", "error", *args], check=True)


def _ffprobe_frames(path: Path) -> int:
    ffprobe = shutil.which("ffprobe")
    if ffprobe is not None:
        out = subprocess.run(
            [
                ffprobe, "-v", "error", "-select_streams", "v:0",
                "-count_frames", "-show_entries", "stream=nb_read_frames",
                "-of", "default=nokey=1:noprint_wrappers=1", str(path),
            ],
            check=True, text=True, capture_output=True,
        )
        return int(out.stdout.strip())
    import cv2  # type: ignore
    cap = cv2.VideoCapture(str(path))
    try:
        return int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    finally:
        cap.release()


def case_paths(case_id: str) -> tuple[Path, Path]:
    sample_id = case_id.split("_", 1)[0]
    sw = sorted((PHYSICSIQ_ROOT / "switch-frames").glob(f"{sample_id}_*.jpg"))
    cond = sorted((PHYSICSIQ_ROOT / "split-videos" / "conditioning" / "24FPS").glob(f"{sample_id}_*.mp4"))
    if not sw or not cond:
        raise FileNotFoundError(f"no inputs found for case {case_id} under {PHYSICSIQ_ROOT}")
    return sw[0], cond[0]


def build_i2v_input_jsonl(case_id: str, dst_jsonl: Path) -> Path:
    prompt = I2V_PROMPTS[case_id]
    switch_frame, _ = case_paths(case_id)
    row = {
        "aspect_ratio": "16,9",
        "fps": 24,
        "guidance": 3.0,
        "guidance_interval": [0, 1000],
        "model_mode": "image2video",
        "name": case_id,
        "negative_metadata_mode": "none",
        "negative_prompt": prompt["negative_prompt"],
        "negative_prompt_keep_metadata": False,
        "num_frames": 121,
        "num_outputs": 1,
        "num_steps": 50,
        "prompt": prompt["prompt"],
        "resolution": "720",
        "seed": 0,
        "shift": 4.75,
        "sigma_max": 80.0,
        "vision_path": str(switch_frame),
    }
    dst_jsonl.parent.mkdir(parents=True, exist_ok=True)
    dst_jsonl.write_text(json.dumps(row) + "\n")
    return dst_jsonl


def prepare_v2v_conditioning(src_72f: Path, dst_73f: Path, fps: int = 24) -> Path:
    """Clone-pad the first frame so the official 72-frame clip becomes 73 frames."""
    n_src = _ffprobe_frames(src_72f)
    assert n_src == 72, f"{src_72f}: expected 72 source frames, got {n_src}"
    dst_73f.parent.mkdir(parents=True, exist_ok=True)
    start_duration = 1.0 / float(fps)
    _ffmpeg([
        "-i", str(src_72f),
        "-vf", f"tpad=start_mode=clone:start_duration={start_duration:.9f}",
        "-r", str(fps),
        "-frames:v", "73",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
        str(dst_73f),
    ])
    n_dst = _ffprobe_frames(dst_73f)
    assert n_dst == 73, f"{dst_73f}: expected 73 frames after pad, got {n_dst}"
    return dst_73f


def build_v2v_input_jsonl(case_id: str, conditioning_73f: Path, dst_jsonl: Path) -> Path:
    prompt = V2V_PROMPTS[case_id]
    row = {
        "aspect_ratio": "16,9",
        "condition_frame_indexes_vision": list(range(19)),
        "fps": 24,
        "guidance": 6.0,
        "model_mode": "video2video",
        "name": case_id,
        "negative_metadata_mode": "same",
        "negative_prompt": prompt["negative_prompt"],
        "negative_prompt_keep_metadata": True,
        "normalize_cfg": False,
        "num_frames": 193,
        "num_steps": 50,
        "prompt": prompt["prompt"],
        "resolution": "720",
        "seed": 0,
        "shift": 5.0,
        "sigma_max": 80.0,
        "vision_path": str(conditioning_73f),
    }
    dst_jsonl.parent.mkdir(parents=True, exist_ok=True)
    dst_jsonl.write_text(json.dumps(row) + "\n")
    return dst_jsonl


def stage_i2v_last120(src_mp4: Path, dst_mp4: Path, fps: int = 24) -> Path:
    """Drop the conditioning frame at index 0 from a 121-frame output, keep the last 120."""
    n_src = _ffprobe_frames(src_mp4)
    assert n_src >= 121, f"{src_mp4}: expected at least 121 frames, got {n_src}"
    start_frame = n_src - 120
    dst_mp4.parent.mkdir(parents=True, exist_ok=True)
    _ffmpeg([
        "-i", str(src_mp4),
        "-vf", f"trim=start_frame={start_frame},setpts=PTS-STARTPTS,fps={fps}",
        "-frames:v", "120",
        "-c:v", "libx264", "-preset", "ultrafast", "-crf", "23",
        "-pix_fmt", "yuv420p", "-an",
        str(dst_mp4),
    ])
    assert _ffprobe_frames(dst_mp4) == 120
    return dst_mp4


def stage_v2v_omit73(src_mp4: Path, dst_mp4: Path, fps: int = 24, omit: int = 73) -> Path:
    """Drop the 73-frame conditioning prefix from a 193-frame V2V output, keep the next 120."""
    n_src = _ffprobe_frames(src_mp4)
    assert n_src >= omit + 120, f"{src_mp4}: expected >= {omit + 120} frames, got {n_src}"
    dst_mp4.parent.mkdir(parents=True, exist_ok=True)
    _ffmpeg([
        "-i", str(src_mp4),
        "-vf", f"select='gte(n,{omit})',setpts=N/({fps}*TB)",
        "-frames:v", "120",
        "-r", str(fps),
        "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
        str(dst_mp4),
    ])
    assert _ffprobe_frames(dst_mp4) == 120
    return dst_mp4


def display_video(path: Path, width: int = 480) -> None:
    from IPython.display import Video, display
    display(Video(filename=str(path), embed=True, width=width))

## 9. I2V (Image-to-Video)

**Recipe**

- **Conditioning**: the single switch-frame JPG. The model loads it as the first generated frame.
- **Output length**: 121 frames at 24 fps (~5.04 s total). Frame 0 is the conditioning frame; frames 1–120 are the 5 seconds of generated content that PhysicsIQ scores.
- **Sampling**: `num_steps=50`, `guidance=3.0`, `guidance_interval=[0, 1000]`, `shift=4.75`, `sigma_max=80.0`, `seed=0`.
- **Prompts**: verbose, timeline-structured per-case prompts shipped in `assets/i2v_prompts.json`.
- **Cropping for scoring**: drop frame 0 (the conditioning frame), keep frames 1..120. PhysicsIQ scores the 120 generated frames.

### Build the I2V input JSONL for the demo case

In [ ]:
i2v_run_dir = PHYSICSIQ_OUTPUT_ROOT / "i2v" / DEMO_CASE_ID
i2v_run_dir.mkdir(parents=True, exist_ok=True)

i2v_input_jsonl = i2v_run_dir / "input.jsonl"
i2v_output_dir = i2v_run_dir / "raw"
i2v_output_dir.mkdir(parents=True, exist_ok=True)

build_i2v_input_jsonl(DEMO_CASE_ID, i2v_input_jsonl)

os.environ["I2V_INPUT"] = str(i2v_input_jsonl)
os.environ["I2V_OUTPUT_DIR"] = str(i2v_output_dir)

print("I2V_INPUT =", i2v_input_jsonl)
print("I2V_OUTPUT_DIR =", i2v_output_dir)
print()
print("first row preview:")
print(i2v_input_jsonl.read_text()[:600], "...")

### Run I2V inference

We use the **latency** parallelism preset with context-parallel sharding across all visible GPUs (`--cp-size=$COSMOS3_NUM_GPUS`). For a single-case run on 4×H100, this completes in roughly 1–2 minutes.

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_INPUT" \
  -o "$I2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

### Stage the I2V output for PhysicsIQ scoring (drop frame 0, keep last 120)

In [ ]:
i2v_raw_mp4 = next((i2v_output_dir).rglob("vision.mp4"))
print("raw output:", i2v_raw_mp4, "frames=", _ffprobe_frames(i2v_raw_mp4))

i2v_staged_mp4 = i2v_run_dir / "staged" / f"{DEMO_CASE_ID}.mp4"
stage_i2v_last120(i2v_raw_mp4, i2v_staged_mp4)
print("staged 120-frame clip:", i2v_staged_mp4)
display_video(i2v_staged_mp4)

## 10. V2V (Video-to-Video)

**Recipe**

- **Conditioning**: a **73-frame** mp4 at 24 fps. The official PhysicsIQ benchmark ships 72-frame conditioning clips; the V2V model expects exactly 73 frames, so we clone-pad the first frame to add one frame at the start. The model uses the first 19 frames of latent (the `condition_frame_indexes_vision=[0..18]` field) as the conditional signal.
- **Output length**: 193 frames at 24 fps. The first 73 frames replay the (padded) conditioning prefix; frames 73..192 are the predicted 5 seconds.
- **Sampling**: `num_steps=50`, `guidance=6.0` (no `guidance_interval`), `shift=5.0`, `sigma_max=80.0`, `seed=0`.
- **Prompts**: structured JSON-style prompts shipped in `assets/v2v_prompts.json` (per-case `task`, `camera`, `goal`, `conditional_video`, `applicable_physics_laws`, `predicted_timeline`).
- **Cropping for scoring**: drop the first 73 frames (conditioning prefix), keep frames 73..192. PhysicsIQ scores the 120 generated frames.


### Pad the 72-frame conditioning clip to 73 frames

In [ ]:
_, conditioning_72f = case_paths(DEMO_CASE_ID)
v2v_run_dir = PHYSICSIQ_OUTPUT_ROOT / "v2v" / DEMO_CASE_ID
v2v_run_dir.mkdir(parents=True, exist_ok=True)
conditioning_73f = v2v_run_dir / "conditioning_73f.mp4"

prepare_v2v_conditioning(conditioning_72f, conditioning_73f)
print("73-frame conditioning ready:", conditioning_73f)
display_video(conditioning_73f)

### Build the V2V input JSONL for the demo case

In [ ]:
v2v_input_jsonl = v2v_run_dir / "input.jsonl"
v2v_output_dir = v2v_run_dir / "raw"
v2v_output_dir.mkdir(parents=True, exist_ok=True)

build_v2v_input_jsonl(DEMO_CASE_ID, conditioning_73f, v2v_input_jsonl)

os.environ["V2V_INPUT"] = str(v2v_input_jsonl)
os.environ["V2V_OUTPUT_DIR"] = str(v2v_output_dir)

print("V2V_INPUT =", v2v_input_jsonl)
print("V2V_OUTPUT_DIR =", v2v_output_dir)

### Run V2V inference

Same parallelism flags as I2V. V2V output is 193 frames, so this takes ~2–3× longer than I2V on 4×H100.

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_V2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$V2V_INPUT" \
  -o "$V2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

### Stage the V2V output for PhysicsIQ scoring (omit first 73, keep next 120)

In [ ]:
v2v_raw_mp4 = next((v2v_output_dir).rglob("vision.mp4"))
print("raw output:", v2v_raw_mp4, "frames=", _ffprobe_frames(v2v_raw_mp4))

v2v_staged_mp4 = v2v_run_dir / "staged" / f"{DEMO_CASE_ID}.mp4"
stage_v2v_omit73(v2v_raw_mp4, v2v_staged_mp4)
print("staged 120-frame clip:", v2v_staged_mp4)
display_video(v2v_staged_mp4)

## 11. (Optional) Run All 198 Cases

The cells below mirror the demo flow but iterate over every case in the benchmark. Each task takes roughly:

| Task | Per case (4×H100) | Full 198 | Disk |
| --- | --- | --- | --- |
| I2V | ~1.5 min | ~5 h | ~16 GB raw + ~360 MB staged |
| V2V | ~3 min | ~10 h | ~24 GB raw + ~360 MB staged |

Run them only if you intend to compute the full PhysicsIQ score.

In [ ]:
import time

RUN_FULL_198 = False  # set to True to enable the full sweep
TASKS_TO_RUN = ("i2v", "v2v")  # subset to ("i2v",) or ("v2v",) if you only need one

if RUN_FULL_198:
    case_ids = sorted(I2V_PROMPTS.keys())
    assert len(case_ids) == 198

    if "i2v" in TASKS_TO_RUN:
        i2v_inputs_dir = PHYSICSIQ_OUTPUT_ROOT / "i2v_full" / "inputs"
        i2v_raw_dir = PHYSICSIQ_OUTPUT_ROOT / "i2v_full" / "raw"
        i2v_staged_dir = PHYSICSIQ_OUTPUT_ROOT / "i2v_full" / "staged"
        i2v_inputs_dir.mkdir(parents=True, exist_ok=True)
        i2v_raw_dir.mkdir(parents=True, exist_ok=True)
        i2v_staged_dir.mkdir(parents=True, exist_ok=True)

        all_jsonl = i2v_inputs_dir / "all_198.jsonl"
        with all_jsonl.open("w") as fp:
            for case_id in case_ids:
                switch_frame, _ = case_paths(case_id)
                prompt = I2V_PROMPTS[case_id]
                fp.write(json.dumps({
                    "aspect_ratio": "16,9", "fps": 24, "guidance": 3.0,
                    "guidance_interval": [0, 1000], "model_mode": "image2video",
                    "name": case_id, "negative_metadata_mode": "none",
                    "negative_prompt": prompt["negative_prompt"],
                    "negative_prompt_keep_metadata": False,
                    "num_frames": 121, "num_outputs": 1, "num_steps": 50,
                    "prompt": prompt["prompt"], "resolution": "720",
                    "seed": 0, "shift": 4.75, "sigma_max": 80.0,
                    "vision_path": str(switch_frame),
                }) + "\n")

        os.environ["I2V_FULL_INPUT"] = str(all_jsonl)
        os.environ["I2V_FULL_OUTPUT_DIR"] = str(i2v_raw_dir)
        print("I2V_FULL_INPUT =", all_jsonl)
        print("Run the next bash cell to generate all 198 I2V outputs, then call stage_i2v_last120 on each.")

    if "v2v" in TASKS_TO_RUN:
        v2v_inputs_dir = PHYSICSIQ_OUTPUT_ROOT / "v2v_full" / "inputs"
        v2v_cond_dir = PHYSICSIQ_OUTPUT_ROOT / "v2v_full" / "conditioning_73f"
        v2v_raw_dir = PHYSICSIQ_OUTPUT_ROOT / "v2v_full" / "raw"
        v2v_staged_dir = PHYSICSIQ_OUTPUT_ROOT / "v2v_full" / "staged"
        for path in (v2v_inputs_dir, v2v_cond_dir, v2v_raw_dir, v2v_staged_dir):
            path.mkdir(parents=True, exist_ok=True)

        # Pad all 198 conditioning clips.
        for case_id in case_ids:
            _, conditioning_72f = case_paths(case_id)
            dst = v2v_cond_dir / f"{case_id}.mp4"
            if not dst.exists():
                prepare_v2v_conditioning(conditioning_72f, dst)

        all_jsonl = v2v_inputs_dir / "all_198.jsonl"
        with all_jsonl.open("w") as fp:
            for case_id in case_ids:
                prompt = V2V_PROMPTS[case_id]
                fp.write(json.dumps({
                    "aspect_ratio": "16,9",
                    "condition_frame_indexes_vision": list(range(19)),
                    "fps": 24, "guidance": 6.0, "model_mode": "video2video",
                    "name": case_id, "negative_metadata_mode": "same",
                    "negative_prompt": prompt["negative_prompt"],
                    "negative_prompt_keep_metadata": True, "normalize_cfg": False,
                    "num_frames": 193, "num_steps": 50,
                    "prompt": prompt["prompt"], "resolution": "720",
                    "seed": 0, "shift": 5.0, "sigma_max": 80.0,
                    "vision_path": str(v2v_cond_dir / f"{case_id}.mp4"),
                }) + "\n")

        os.environ["V2V_FULL_INPUT"] = str(all_jsonl)
        os.environ["V2V_FULL_OUTPUT_DIR"] = str(v2v_raw_dir)
        print("V2V_FULL_INPUT =", all_jsonl)
        print("Run the next bash cell to generate all 198 V2V outputs, then call stage_v2v_omit73 on each.")
else:
    print("Set RUN_FULL_198 = True above and re-run to enable the full 198-case sweep.")

### Generate all 198 I2V outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${I2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_FULL_198 = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_FULL_INPUT" \
  -o "$I2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

### Generate all 198 V2V outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${V2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_FULL_198 = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_V2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$V2V_FULL_INPUT" \
  -o "$V2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

### Stage all 198 outputs

In [ ]:
if RUN_FULL_198:
    if "i2v" in TASKS_TO_RUN:
        for raw in sorted(Path(os.environ["I2V_FULL_OUTPUT_DIR"]).rglob("vision.mp4")):
            case_id = raw.parent.name
            dst = Path(os.environ["I2V_FULL_OUTPUT_DIR"]).parent / "staged" / f"{case_id}.mp4"
            if not dst.exists():
                stage_i2v_last120(raw, dst)
        print("I2V staged dir:", Path(os.environ["I2V_FULL_OUTPUT_DIR"]).parent / "staged")
    if "v2v" in TASKS_TO_RUN:
        for raw in sorted(Path(os.environ["V2V_FULL_OUTPUT_DIR"]).rglob("vision.mp4")):
            case_id = raw.parent.name
            dst = Path(os.environ["V2V_FULL_OUTPUT_DIR"]).parent / "staged" / f"{case_id}.mp4"
            if not dst.exists():
                stage_v2v_omit73(raw, dst)
        print("V2V staged dir:", Path(os.environ["V2V_FULL_OUTPUT_DIR"]).parent / "staged")

## 12. (Optional) Run the Official PhysicsIQ Scorer

The official scorer compares generated 120-frame clips against ground-truth motion masks and writes a per-case CSV plus an overall PhysicsIQ score. It expects staged clips named exactly like the case ids (no trailing suffix).

Refer to https://github.com/google-deepmind/physics-IQ-benchmark for the canonical scoring instructions. The cell below shows the call we used to produce the reference numbers.

In [ ]:
%%bash
set -euo pipefail

# Set STAGED_DIR to the staged folder you want to score.
# Demo run (single case I2V): $PHYSICSIQ_OUTPUT_ROOT/i2v/<CASE_ID>/staged
# Demo run (single case V2V): $PHYSICSIQ_OUTPUT_ROOT/v2v/<CASE_ID>/staged
# Full 198-case I2V sweep:    $PHYSICSIQ_OUTPUT_ROOT/i2v_full/staged  (default)
STAGED_DIR="${STAGED_DIR:-$PHYSICSIQ_OUTPUT_ROOT/i2v_full/staged}"
RESULTS_DIR="${RESULTS_DIR:-$PHYSICSIQ_OUTPUT_ROOT/scoring}"
PHYSICSIQ_REPO="$(dirname "$PHYSICSIQ_ROOT")/physics-IQ-benchmark-repo"

if [ ! -d "$STAGED_DIR" ]; then
  echo "No staged clips at $STAGED_DIR. Run the staging cells first."
  exit 0
fi
if [ ! -d "$PHYSICSIQ_REPO" ]; then
  echo "Clone the PhysicsIQ scorer repo first (the download cell does this)."
  exit 1
fi

mkdir -p "$RESULTS_DIR"
cd "$PHYSICSIQ_REPO"
python3 ./code/run_physics_iq.py \
  --generated_videos_dir "$STAGED_DIR" \
  --results_dir "$RESULTS_DIR" \
  --physics_iq_dir "$PHYSICSIQ_ROOT" \
  --fps 24